In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Question :** Given us Customers dataframe, We need to have separate record for each and every Customer.

In [0]:
data = [("John,Smith", "Canada"), ("Mike,David", "USA")]
df = spark.createDataFrame(data,["Names","Country"])

In [0]:
display(df)

In [0]:
%python
df_final = df.withColumn("New_Names", explode_outer(split(col("Names"),","))).select("New_Names","Country")
display(df_final)

Given us Players and Countries dataframe, We need to find total numbers of 50s and 100s for palyers.


In [0]:
from pyspark.sql.functions import *

In [0]:
schema = StructType([
    StructField("player", StringType(), True),
    StructField("runs", IntegerType(), True),
    StructField("50s/100s", StringType(), True)
])

data = [("Sachin-IND", 18694, "93/49"), ("Ricky-AUS", 11274, "66/31"),("Lara-WI", 10222, "45/21"),("Rahul-IND", 10355, "95/11"),("Jhonty-SA", 7051, "43/5"),("Hayden-AUS", 8722, "67/19")]
players_df = spark.createDataFrame(data, schema)

data1 = [("IND", "India"), ("AUS", "Australia"), ("WI", "WestIndies"), ("SA", "SouthAfrica")]
countries_df = spark.createDataFrame(data1,["SRT","country"])
display(players_df)
display(countries_df)

In [0]:
players_df2 = players_df.withColumn("player_srt",split(col("player"),"-")[1]).withColumn("player",split(col("player"),"-")[0]).withColumn("50s",split(col("50s/100s"),"/")[0]).withColumn("100s",split(col("50s/100s"),"/")[1])
display(players_df2)

In [0]:
joined_df = players_df2.join(countries_df,players_df2.player_srt == countries_df.srt,"left").withColumn("sum",(col("50s").cast('int') + col("100s").cast('int')))
final_df = joined_df.select("player","runs","sum","country","SRT").where("sum >= 90")

In [0]:
display(final_df)

Tiger Analytics
Question : Calculate % difference between sales in quarter 1 and quarter 2 for each year

In [0]:
data = [
    ("2010-01-02", 500),
    ("2010-02-03", 1000),
    ("2010-03-04", 1000),
    ("2010-04-05", 1000),
    ("2010-05-06", 1500),
    ("2010-06-07", 1000),
    ("2010-07-08", 1000),
    ("2010-08-09", 1000),
    ("2011-10-10", 1000),
    ("2011-01-02", 500),
    ("2011-02-03", 1000),
    ("2011-03-04", 1000),
    ("2011-04-05", 1000),
    ("2011-05-06", 1550),
    ("2011-06-07", 1100),
    ("2011-07-08", 1100),
    ("2011-08-09", 1000),
]
schema = ["date", "sales"]
df = spark.createDataFrame(data, schema).withColumn("date",to_date(col("date"),"yyyy-MM-dd")).withColumn("year",year(col("date"))).withColumn("month",month(col("date"))).withColumn("quarter",quarter(col("date")))
display(df)

In [0]:
display(df.withColumn("quarter_self",when((col('month') >=1) & (col('month') <=3), 1).when((col('month') >=4) & (col('month')<=6), 2).when((col("month") >=7 )& (col("month") <=9), 3).when((col("month") >=10) & (col("month") <=12), 4)))

In [0]:
interm_df = df.where("quarter in (1,2)").groupBy("quarter","year").agg(sum("sales").alias("quarter_sales"))
display(interm_df)

In [0]:
from pyspark.sql.window import Window

# final_df = interm_df.withColumn("%Diff",expr("round((quarter_sales -lag(quarter_sales,1) over(partition by year order by quarter))*100.0/NULLIF(lag(quarter_sales, 1) over (partition by year order by quarter),0),2")))


final_df = interm_df.withColumn("%Diff",
    expr("round((quarter_sales - lag(quarter_sales, 1) over (partition by year order by quarter)) * 100.0 / NULLIF(lag(quarter_sales, 1) over (partition by year order by quarter), 0), 2)")
)

In [0]:
display(final_df)

Another way of solving this:

In [0]:
data = [
    ("2010-01-02", 500),
    ("2010-02-03", 1000),
    ("2010-03-04", 1000),
    ("2010-04-05", 1000),
    ("2010-05-06", 1500),
    ("2010-06-07", 1000),
    ("2010-07-08", 1000),
    ("2010-08-09", 1000),
    ("2011-10-10", 1000),
    ("2011-01-02", 500),
    ("2011-02-03", 1000),
    ("2011-03-04", 1000),
    ("2011-04-05", 1000),
    ("2011-05-06", 1550),
    ("2011-06-07", 1100),
    ("2011-07-08", 1100),
    ("2011-08-09", 1000),
]
schema = ["date", "sales"]
df = spark.createDataFrame(data, schema).withColumn("date",to_date(col("date"),"yyyy-MM-dd")).withColumn("year",year(col("date"))).withColumn("month",month(col("date"))).withColumn("quarter",quarter(col("date")))
df_agg = df.groupBy("year","quarter").agg(sum("sales").alias("quarter_sales"))
display(df_agg)

In [0]:
pivoted_sales = df_agg.groupBy("year").pivot("quarter",[1,2]).agg(sum("quarter_sales")).withColumnRenamed("1","Q1").withColumnRenamed("2","Q2")
display(pivoted_sales)

In [0]:
flights_data = [
    (101, 'AI301', 'Chennai', 'Bangalore', '2024-01-15 08:30:00'),
    (101, 'SG205', 'Bangalore', 'Goa', '2024-01-15 14:20:00'),
    (101, 'UK847', 'Goa', 'Pune', '2024-01-17 11:15:00'),
    (102, 'SJ612', 'Kolkata', 'Bhubaneswar', '2024-01-20 09:45:00'),
    (102, 'AI108', 'Bhubaneswar', 'Visakhapatnam', '2024-01-21 16:30:00'),
    (102, 'UK203', 'Visakhapatnam', 'Tirupati', '2024-01-23 10:55:00'),
    (103, 'SG450', 'Jaipur', 'Udaipur', '2024-01-25 07:20:00'),
    (103, 'AI789', 'Udaipur', 'Ahmedabad', '2024-01-26 18:40:00'),
    (104, 'UK167', 'Chandigarh', 'Amritsar', '2024-02-01 12:10:00'),
    (104, 'SJ893', 'Amritsar', 'Jammu', '2024-02-02 15:25:00'),
    (104, 'AI445', 'Jammu', 'Srinagar', '2024-02-03 09:35:00'),
    (105, 'SG721', 'Lucknow', 'Varanasi', '2024-02-05 06:15:00'),
    (105, 'UK356', 'Varanasi', 'Patna', '2024-02-05 19:50:00')
]

_schema = "cust_id int, flight_id string, origin string, destination string, flight_timestamp string"

df_flight = spark.createDataFrame(data=flights_data, schema=_schema).withColumn("flight_timestamp",to_timestamp(col("flight_timestamp"),"yyyy-MM-dd HH:mm:ss"))

In [0]:
display(df_flight)

In [0]:
from pyspark.sql.window import Window 

window = Window.partitionBy(col("cust_id")).orderBy(col("flight_timestamp").asc())

df = df_flight.withColumn("rank",row_number().over(window))


In [0]:
display(df)

In [0]:
df2 = df.groupBy("cust_id").agg(min(col("rank")).alias("start"),max(col("rank")).alias("end"))

In [0]:
df3 = df.join(df2,df.cust_id == df2.cust_id,"left")

In [0]:
display(df3)